## MLegS Post-Processing Visualization

This notebook is designed to read and visualize 2D field data generated by the `postproc` executable from the MLegS simulation package.

### Functionality:
1.  **Reads Collocation Points**: Loads the physical grid coordinates (`r`, `theta`, `z`) from the `.info` files in the output directory.
2.  **Identifies Data Files**: Scans the output directory for 2D slice data files (e.g., `velR_RTplane_001.dat`, `vorZ_RZplane_010.dat`).
3.  **Parses Filenames**: Extracts metadata from filenames, such as the field type, slice orientation, and snapshot index.
4.  **Loads and Reshapes Data**: Reads the raw data, which is stored as pairs of real and imaginary components, and reshapes it into the correct 2D complex array corresponding to the slice.
5.  **Visualizes Fields**: Creates contour plots for the real part of the selected field data. It handles both R-Theta and R-Z plane visualizations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import re
from ipywidgets import interact, Dropdown, IntSlider, VBox

### Load Physical Collocation Points

We load the radial, azimuthal, and axial grid points from the `.info` files. These are necessary to correctly plot the data in physical space.

In [ ]:
def load_collocation_points(directory):
    """Loads r, theta, and z grid points from .info files."""
    try:
        r_pts = np.loadtxt(os.path.join(directory, 'r_colloc_pts.info'))
        th_pts = np.loadtxt(os.path.join(directory, 't_colloc_pts.info'))
        z_pts = np.loadtxt(os.path.join(directory, 'z_colloc_pts.info'))
        print(f"Loaded grid points:")
        print(f"  - Radial (r): {len(r_pts)} points")
        print(f"  - Azimuthal (theta): {len(th_pts)} points")
        print(f"  - Axial (z): {len(z_pts)} points")
        return r_pts, th_pts, z_pts
    except FileNotFoundError as e:
        print(f"Error loading grid files: {e}")
        print("Please ensure 'r_colloc_pts.info', 't_colloc_pts.info', and 'z_colloc_pts.info' are in the output directory.")
        return None, None, None

In [ ]:
output_dir = '../../output/'
r_pts, th_pts, z_pts = load_collocation_points(output_dir)

### Discover and Parse Data Files

This section scans the output directory for valid 2D slice data files and parses their names to understand what each file contains.

In [ ]:
def find_data_files(directory):
    """Finds and parses 2D slice data files in the given directory."""
    # Regex to match filenames like 'velR_RTplane_001.dat' or 'vorZ_RZplane_010.dat'
    # It captures the field name, slice type, and index.
    pattern = re.compile(r"^(?P<field>\w+?)_(?P<slice_type>RTplane|RZplane)_(?P<index>\d+)\.dat$")
    
    file_info = []
    print(f"\nScanning for data files in: {directory}")
    for filename in sorted(os.listdir(directory)):
        match = pattern.match(filename)
        if match:
            info = match.groupdict()
            info['filename'] = filename
            file_info.append(info)
            
    if not file_info:
        print("No 2D slice data files found. Make sure 'POSTPROCESS%SLICEINT' is not 999 and filenames match the expected pattern.")
    else:
        print(f"Found {len(file_info)} data files.")
        for info in file_info:
            print(f"  - {info['filename']}: Field = {info['field']}, Slice = {info['slice_type']}, Index = {info['index']}")
        
    return file_info

In [ ]:
data_files = find_data_files(output_dir)

### Data Loading and Processing

The core function to read a `.dat` file. The Fortran `MSAVEC` routine writes complex numbers as two consecutive real numbers (real part, then imaginary part). This function reads that flat list of numbers and reconstructs the 2D complex-valued array.

In [ ]:
def load_and_reshape_data(filepath, slice_type, grid_shapes):
    """
    Loads data from a file and reshapes it into the correct 2D real-valued array.
    - For RTplane, the data is real across all theta points.
    - For RZplane, the z-direction is periodic. The data is saved for NZ points,
      but the coordinates have NZ+1 points. We pad the data array by copying
      the first z-slice to the end to close the domain for plotting.
      
    Data storage convention:
    - For 2D data: slowest dimension is radial (r), fastest is the other dimension
    - For 3D data: slowest is z, then r, then theta (fastest)
    """
    try:
        # The data is a single line of text with space-separated numbers.
        raw_data = np.loadtxt(filepath).T
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None

    # Determine the correct shape based on the slice type
    nr, nth, nz = grid_shapes
    if slice_type == 'RTplane':
        # For RT plane: data layout is nr x (nth-1) with radial as slowest dimension
        # The first nth values are for r=0, next nth for r=1, etc.
        expected_shape = (nth - 1, nr)  # Shape of raw data as stored
        real_data_unpadded = raw_data
        if real_data_unpadded.shape != expected_shape:
            print(f"Error: Data size mismatch for {filepath}.")
            print(f"  - Expected {expected_shape} real values, but found {real_data_unpadded.shape}.")
            print(f"  - Expected shape for storage: {expected_shape}")
            return None
        
        real_data = np.vstack((real_data_unpadded, real_data_unpadded[0:1, :]))  # Pad theta by appending first column

        return real_data.T
        
    elif slice_type == 'RZplane':
        # For RZ plane: data layout is nr x (nz-1) with radial as slowest dimension
        # The first (nz-1) values are for r=0, next (nz-1) for r=1, etc.
        expected_shape = (nz - 1, nr)  # Shape of complex data as stored
        
        if raw_data.size % 2 != 0:
            print(f"Warning: RZ data in {filepath} has an odd number of elements.")
            return None
            
        # Reconstruct complex numbers and take the real part for the slice.
        complex_data = raw_data[::2] + 1j * raw_data[1::2]
        real_data_unpadded = np.real(complex_data)
        if real_data_unpadded.shape != expected_shape:
            print(f"Error: Data size mismatch for {filepath}.")
            print(f"  - Expected {expected_shape} real values, but found {real_data_unpadded.shape}.")
            print(f"  - Expected shape for storage: {expected_shape}")
            return None
        
        real_data = np.vstack((real_data_unpadded, real_data_unpadded[0:1, :])) # Pad z by appending first column

        return real_data.T # Return padded data with shape (nr, nz)
    else:
        return None


In [ ]:
info = data_files[0]
filepath = os.path.join(output_dir, info['filename'])
grid_shapes = (len(r_pts), len(th_pts), len(z_pts))

# Load and process the data
data = load_and_reshape_data(filepath, info['slice_type'], grid_shapes)
print(data.shape)

data[:,0]

In [ ]:
def plot_rt_slice(data, r_coords, th_coords, title, r_max = 10.0, use_log_scale=True):
    """Creates a polar contour plot for an R-Theta slice."""
    if data is None:
        return

    # Filter data to only include points within r_max
    r_mask = r_coords <= r_max
    r_filtered = r_coords[r_mask]
    data_filtered = data[r_mask, :]

    # Create a meshgrid for polar plot
    R, TH = np.meshgrid(r_filtered, th_coords)

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})
    
    if use_log_scale:
        # Apply symmetric logarithmic transformation
        # For values with same sign, use sign(x) * log10(1 + |x|/threshold)
        max_abs_val = np.max(np.abs(data_filtered))
        threshold = max_abs_val * 1e-6  # Threshold to avoid log(0)
        
        # Transform data: sign(x) * log10(1 + |x|/threshold)
        data_log = np.sign(data_filtered.T) * np.log10(1 + np.abs(data_filtered.T) / threshold)
        
        # Set symmetric color limits for log-transformed data
        max_log_val = np.max(np.abs(data_log))
        vmin, vmax = -max_log_val, max_log_val
        
        contour = ax.pcolormesh(TH, R, data_log, shading='auto', cmap='RdBu_r', 
                               vmin=vmin, vmax=vmax)
        
        # Create custom colorbar with original values
        cbar = fig.colorbar(contour, ax=ax, orientation='vertical', label='Field Value')
        
        # Create custom tick labels showing original values
        n_ticks = 7
        log_ticks = np.linspace(vmin, vmax, n_ticks)
        original_ticks = np.sign(log_ticks) * threshold * (10**np.abs(log_ticks) - 1)
        
        cbar.set_ticks(log_ticks)
        cbar.set_ticklabels([f'{val:.2e}' for val in original_ticks])
        
    else:
        # Standard linear color scale
        max_abs_val = np.max(np.abs(data_filtered))
        vmin, vmax = -max_abs_val, max_abs_val
        
        contour = ax.pcolormesh(TH, R, data_filtered.T, shading='auto', cmap='RdBu_r', 
                               vmin=vmin, vmax=vmax)
        
        fig.colorbar(contour, ax=ax, orientation='vertical', label='Field Value')
    
    ax.set_title(title, va='bottom', fontsize=14)
    ax.set_ylim(0, r_max) # Set radial limit to r_max
    ax.set_xlabel('$\theta$')
    ax.set_ylabel('$r$', labelpad=20)
    plt.show()

def plot_rz_slice(data, r_coords, z_coords, title, r_max = 10.0, use_log_scale=True):
    """Creates a Cartesian contour plot for an R-Z slice."""
    if data is None:
        return

    # Filter data to only include points within r_max
    r_mask = r_coords <= r_max
    r_filtered = r_coords[r_mask]
    data_filtered = data[r_mask, :]

    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Create a meshgrid for the plot with r as x-axis and z as y-axis
    R, Z = np.meshgrid(r_filtered, z_coords)

    if use_log_scale:
        # Apply symmetric logarithmic transformation
        max_abs_val = np.max(np.abs(data_filtered))
        threshold = max_abs_val * 1e-6  # Threshold to avoid log(0)
        
        # Transform data: sign(x) * log10(1 + |x|/threshold)
        data_log = np.sign(data_filtered.T) * np.log10(1 + np.abs(data_filtered.T) / threshold)
        
        # Set symmetric color limits for log-transformed data
        max_log_val = np.max(np.abs(data_log))
        vmin, vmax = -max_log_val, max_log_val
        
        contour = ax.pcolormesh(R, Z, data_log, shading='auto', cmap='RdBu_r',
                               vmin=vmin, vmax=vmax)
        
        # Create custom colorbar with original values
        cbar = fig.colorbar(contour, ax=ax, label='Field Value')
        
        # Create custom tick labels showing original values
        n_ticks = 7
        log_ticks = np.linspace(vmin, vmax, n_ticks)
        original_ticks = np.sign(log_ticks) * threshold * (10**np.abs(log_ticks) - 1)
        
        cbar.set_ticks(log_ticks)
        cbar.set_ticklabels([f'{val:.2e}' for val in original_ticks])
        
    else:
        # Standard linear color scale
        max_abs_val = np.max(np.abs(data_filtered))
        vmin, vmax = -max_abs_val, max_abs_val
        
        contour = ax.pcolormesh(R, Z, data_filtered.T, shading='auto', cmap='RdBu_r',
                               vmin=vmin, vmax=vmax)
        
        fig.colorbar(contour, ax=ax, label='Field Value')
    
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Radial Coordinate (r)')
    ax.set_ylabel('Axial Coordinate (z)')
    ax.set_xlim(0, r_max) # Set radial limit to r_max
    ax.set_aspect('equal')
    plt.show()

### 6. Interactive Visualization

Use the dropdown menus to select the field and snapshot index you want to visualize. The notebook will automatically load the data and generate the appropriate plot.

In [ ]:
def interactive_plotter(file_to_plot):
    """Main function driven by the ipywidgets interact decorator."""
    if not file_to_plot or not all((r_pts is not None, th_pts is not None, z_pts is not None)):
        print("Cannot plot. Check that data and grid files were loaded correctly.")
        return
    
    # Find the corresponding info dictionary
    info = next((item for item in data_files if item['filename'] == file_to_plot), None)
    if not info:
        print(f"Could not find info for {file_to_plot}")
        return

    filepath = os.path.join(output_dir, info['filename'])
    grid_shapes = (len(r_pts), len(th_pts), len(z_pts))

    # Load and process the data
    data = load_and_reshape_data(filepath, info['slice_type'], grid_shapes)

    if data is None:
        print(f"Failed to load or process data for {info['filename']}.")
        return

    # Generate title and call the correct plotting function
    title = f"Field: {info['field']} | Slice: {info['slice_type']} | Index: {info['index']}"
    
    if info['slice_type'] == 'RTplane':
        plot_rt_slice(data, r_pts, th_pts, title, r_max=15, use_log_scale=True)
    elif info['slice_type'] == 'RZplane':
        plot_rz_slice(data, r_pts, z_pts, title, r_max=15, use_log_scale=True)

# Create a list of filenames for the dropdown
if data_files:
    file_options = [f['filename'] for f in data_files]
    interact(interactive_plotter, file_to_plot=Dropdown(options=file_options, description='Select File:'))
else:
    print("\nNo files to display. Run the cells above to scan for data.")